# 🏥 KUH Ameliyat Vaka Hacmi Tahmini — v5 (2020–2026)
### LSTM · Facebook Prophet · LightGBM · Weighted Ensemble
**Kaynak:** Bui et al. (2023) — *IISE Transactions on Healthcare Systems Engineering*

---

## 📊 Veri Seti Profili
| Özellik | Değer |
|---|---|
| **Ameliyat Verisi** | KUH_2020_2026_Ameliyat.xlsx (68,473 kayıt) |
| **Doluluk Verisi** | KUH_Hastane_Doluluk_Oranı.xlsx (Ara 2022 – Kas 2025, günlük) |
| **Eğitim** | Oca 2020 – Ara 2024 · **Validasyon** | Oca 2025 – Ara 2025 |
| **Tahmin** | Oca 2026 – Mar 2026 |

## ✅ v4'e göre değişiklikler (Kapsamlı Tuning — v5)
| # | Alan | Değişiklik | Beklenen etki |
|---|---|---|---|
| 1 | **LSTM** | `learning_rate 1e-3→3e-4`, `clipnorm 1.0→5.0` | Daha stabil convergence |
| 2 | **LSTM** | `recurrent_dropout` parametreye çıkarıldı (0.1) | Ayrı tuning imkânı |
| 3 | **LSTM** | Huber `delta=1.0` parametreye çıkarıldı | Aykırı değer hassasiyeti |
| 4 | **LSTM** | Decoder'a 2. LSTM katmanı eklendi | Daha derin kod çözme |
| 5 | **Scaler** | `MinMaxScaler → RobustScaler` | Pandemi dönemi aykırılıklarına dayanıklı |
| 6 | **GBM** | `sklearn GBM → LightGBM` | 10× hız, daha iyi performans |
| 7 | **Özellik** | Tatil penceresi `±2→±3`, lag `{3,60,90}` eklendi | Daha uzun bağlam |
| 8 | **Özellik** | `EWMA(span=7)` doluluk özelliği eklendi | Son günlere ağırlık |
| 9 | **Özellik** | COVID sınırı parametreye çıkarıldı | Kolayca test edilebilir |
| 10 | **Prophet** | `seasonality_mode→multiplicative`, `yearly_seasonality→15` | Mevsimsel ölçek etkisi |
| 11 | **Ensemble** | Val R²'ye göre ağırlıklı birleştirme eklendi | En iyi modellerin etkisi artar |

---
### ▶️ Kullanım
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Her iki Excel dosyasını sırayla yükle

In [ ]:
# CELL 1 — Bağımlılıkları kur
!pip install prophet openpyxl lightgbm -q
print('✔ Kurulum tamamlandı (prophet · openpyxl · lightgbm)')

In [ ]:
# CELL 2 — Import & Konfigürasyon
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, RepeatVector,
    TimeDistributed, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from prophet import Prophet
from sklearn.preprocessing import RobustScaler          # v5: MinMaxScaler → RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import lightgbm as lgb                                  # v5: sklearn GBM → LightGBM

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  TUNING PANELİ — v5: tüm parametreler tek yerde
# ─────────────────────────────────────────────────────────────────────────────

# — LSTM —
SEED              = 42
LOOKBACK          = 28      # Pencere uzunluğu (gün): 14, 21, 28, 42, 56
HORIZON           = 7
LSTM_UNITS        = 64      # Katman genişliği: 32, 64, 96, 128
DROPOUT           = 0.3     # Dropout oranı: 0.1, 0.2, 0.3, 0.4
RECURRENT_DROPOUT = 0.1     # v5 YENİ — ayrı parametreye çıkarıldı: 0.1, 0.2, 0.3
HUBER_DELTA       = 1.0     # v5 YENİ — Huber δ: 0.5, 1.0, 2.0
EPOCHS            = 200
BATCH_SIZE        = 16
LEARNING_RATE     = 3e-4    # v5: 1e-3 → 3e-4 (daha stabil convergence)
CLIP_NORM         = 5.0     # v5: 1.0 → 5.0  (sıkı kırpmayı gevşet)
EARLY_PATIENCE    = 30      # EarlyStopping: 20, 30, 40
REDUCE_FACTOR     = 0.5     # ReduceLROnPlateau factor: 0.3, 0.5
REDUCE_PATIENCE   = 15      # ReduceLR patience: 10, 15, 20
VAL_SPLIT         = 0.15    # LSTM iç validasyon: 0.10, 0.15, 0.20

# — Özellik mühendisliği —
COVID_END         = '2022-06-01'   # v5 YENİ — pandemi sınırı: 2022-03, 2022-06, 2022-09
HOL_WINDOW        = 3              # v5: ±2 → ±3 (tatil yakınlık penceresi)
DOL_EWMA_SPAN     = 7             # v5 YENİ — EWMA span: 3, 7, 14, 30

# — LightGBM —
LGB_N_ESTIMATORS  = 300    # Ağaç sayısı: 200, 300, 500
LGB_MAX_DEPTH     = 6      # Derinlik: 4, 6, 8 (-1 = sınırsız)
LGB_LR            = 0.05   # Öğrenme hızı: 0.01, 0.03, 0.05, 0.1
LGB_SUBSAMPLE     = 0.8    # Satır örnekleme: 0.6, 0.8, 0.9
LGB_MIN_LEAF      = 5      # Min yaprak örneği: 3, 5, 10
LGB_COLSAMPLE     = 0.8    # v5 YENİ — kolon örnekleme: 0.6, 0.8, 1.0

# — Prophet —
PROPHET_CP_SCALE  = 0.15   # changepoint_prior_scale: 0.05, 0.15, 0.3, 0.5
PROPHET_SEAS_MODE = 'multiplicative'   # v5: 'additive' → 'multiplicative'
PROPHET_SEAS_PRIOR= 10.0   # seasonality_prior_scale
PROPHET_HOL_PRIOR = 15.0   # holidays_prior_scale
PROPHET_YEARLY_N  = 15     # v5: 10 → 15 Fourier bileşeni

# — Dönem sınırları —
TRAIN_END      = '2025-01-01'
VAL_END        = '2026-01-01'
FORECAST_START = '2026-01-01'
FORECAST_END   = '2026-03-31'

np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print(f'✔ TensorFlow {tf.__version__}')
print(f'  GPU: {gpus[0].name if gpus else "Bulunamadı — CPU modu (daha yavaş)"}')
print(f'  LightGBM: {lgb.__version__}')
print()
print('⚙️  Aktif tuning parametreleri:')
print(f'   LSTM        : LOOKBACK={LOOKBACK}  UNITS={LSTM_UNITS}  DROPOUT={DROPOUT}  REC_DROP={RECURRENT_DROPOUT}')
print(f'   Optimizer   : LR={LEARNING_RATE}  CLIP={CLIP_NORM}  HUBER_δ={HUBER_DELTA}')
print(f'   Özellik     : COVID_END={COVID_END}  HOL_WIN=±{HOL_WINDOW}  EWMA_SPAN={DOL_EWMA_SPAN}')
print(f'   LightGBM    : trees={LGB_N_ESTIMATORS}  depth={LGB_MAX_DEPTH}  lr={LGB_LR}  col={LGB_COLSAMPLE}')
print(f'   Prophet     : cp={PROPHET_CP_SCALE}  mode={PROPHET_SEAS_MODE}  yearly_n={PROPHET_YEARLY_N}')

In [ ]:
# CELL 3 — Veri yükle (Ameliyat + Doluluk)
from google.colab import files

print('📂 Lütfen yükleyin: KUH_2020_2026_Ameliyat.xlsx')
uploaded1  = files.upload()
DATA_PATH  = list(uploaded1.keys())[0]
print(f'✔ Yüklendi: {DATA_PATH}')

print()
print('📂 Lütfen yükleyin: KUH_Hastane_Doluluk_Oranı.xlsx')
uploaded2    = files.upload()
DOLULUK_PATH = list(uploaded2.keys())[0]
print(f'✔ Yüklendi: {DOLULUK_PATH}')

In [ ]:
# CELL 4 — Veri yükle, günlük hacme çevir & Doluluk oranı ile birleştir
# v4: Hastane Doluluk Oranı entegrasyonu + mevsimsel impütasyon

# ── 4a: Ameliyat verisi ──────────────────────────────────────────────────────
df_raw = pd.read_excel(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw.iloc[:, 2], errors='coerce').dt.normalize()
df_raw = df_raw.dropna(subset=['date'])

daily_raw  = df_raw.groupby('date').size().reset_index(name='volume')
full_range = pd.date_range(daily_raw['date'].min(), daily_raw['date'].max(), freq='D')
daily = (daily_raw.set_index('date')
                  .reindex(full_range, fill_value=0)
                  .reset_index()
                  .rename(columns={'index': 'date'}))

df_raw['month_key'] = df_raw['date'].dt.to_period('M')
prov_monthly = (df_raw.groupby('month_key')['Primer Doktor Kodu']
                      .nunique().reset_index(name='num_prov'))

print(f'✔ Ameliyat: {len(df_raw):,} kayıt yüklendi')
print(f'  Tarih aralığı : {daily["date"].min().date()} → {daily["date"].max().date()}')
print(f'  Toplam gün    : {len(daily)}  (veri olan: {(daily.volume > 0).sum()})')
print(f'  Günlük hacim  : min={daily.volume.min()}  maks={daily.volume.max()}  ort={daily.volume.mean():.1f}')

# ── 4b: Doluluk verisi yükle ─────────────────────────────────────────────────
df_dol = pd.read_excel(DOLULUK_PATH)
df_dol.columns = ['date', 'doluluk']
df_dol['date']    = pd.to_datetime(df_dol['date']).dt.normalize()
df_dol['doluluk'] = pd.to_numeric(df_dol['doluluk'], errors='coerce')
df_dol = df_dol.dropna().drop_duplicates('date').set_index('date').sort_index()

print(f'\n✔ Doluluk: {len(df_dol):,} gün yüklendi')
print(f'  Tarih aralığı : {df_dol.index.min().date()} → {df_dol.index.max().date()}')
print(f'  Aralık        : {df_dol.doluluk.min():.2f} – {df_dol.doluluk.max():.2f}  (ort={df_dol.doluluk.mean():.3f})')

# ── 4c: Mevsimsel impütasyon ─────────────────────────────────────────────────
# Gerçek veriden aylık ortalama hesapla (ay bazında mevsimsel desen)
dol_monthly_avg = df_dol.copy()
dol_monthly_avg['month'] = dol_monthly_avg.index.month
monthly_avg = dol_monthly_avg.groupby('month')['doluluk'].mean()

# Tüm gün aralığı için doluluk serisi (ameliyat + tahmin dönemi)
all_dates  = pd.date_range(daily['date'].min(), '2026-03-31', freq='D')
dol_full   = pd.Series(index=all_dates, dtype=float, name='doluluk')
dol_full.update(df_dol['doluluk'])
# Eksik günleri aynı aydaki gerçek ortalamayla doldur
for dt in dol_full[dol_full.isna()].index:
    dol_full[dt] = monthly_avg.get(dt.month, monthly_avg.mean())

# Türetilmiş doluluk özellikleri
dol_roll7 = dol_full.rolling(7, min_periods=1, center=True).mean()
dol_delta = dol_full.diff().fillna(0)

# daily DataFrame'e ekle
daily['doluluk']   = daily['date'].map(dol_full).values
daily['dol_roll7'] = daily['date'].map(dol_roll7).values
daily['dol_delta'] = daily['date'].map(dol_delta).values

print(f'\n✔ Doluluk özellikleri daily DataFrame\'e eklendi')
print(f'  Gerçek veri kapsamı   : Ara 2022 – Kas 2025')
print(f'  İmpüte edildi (önce)  : Oca 2020 – Kas 2022  ({(daily["date"] < "2022-12-01").sum()} gün)')
print(f'  İmpüte edildi (sonra) : Kas 2025 – Mar 2026  (tahmin periyodu)')

# ── 4d: EDA Grafikleri ───────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(16, 14))

# Ameliyat hacmi
axes[0].plot(daily['date'], daily['volume'], color='#3B82F6', lw=0.8, alpha=0.8)
axes[0].axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2022-01-01'),
                alpha=0.08, color='red', label='Pandemi dönemi')
axes[0].axvspan(pd.Timestamp(TRAIN_END), pd.Timestamp(VAL_END),
                alpha=0.08, color='orange', label='Validasyon 2025')
axes[0].set_title('Günlük Ameliyat Vaka Hacmi (2020–2026)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Vaka'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Doluluk oranı (gerçek + impüte)
dol_vis  = dol_full[(dol_full.index >= daily['date'].min()) &
                    (dol_full.index <= daily['date'].max())]
axes[1].fill_between(dol_vis.index, dol_vis.values, alpha=0.2, color='#8B5CF6')
axes[1].plot(dol_vis.index, dol_vis.values,
             color='#8B5CF6', lw=0.8, alpha=0.9, label='Doluluk (tüm dönem)')
axes[1].plot(dol_roll7[dol_roll7.index.isin(daily['date'])].index,
             dol_roll7[dol_roll7.index.isin(daily['date'])].values,
             color='#4C1D95', lw=2, label='7-günlük ort.')
axes[1].axvspan(daily['date'].min(), pd.Timestamp('2022-11-30'),
                alpha=0.08, color='gray', label='Mevsimsel impütasyon (öncesi)')
axes[1].set_title('Hastane Doluluk Oranı — Gerçek + Mevsimsel İmpütasyon (v4)',
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Doluluk Oranı'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# Yıllık ortalama hacim
daily['year'] = daily['date'].dt.year
yr_avg = daily[daily['volume']>0].groupby('year')['volume'].mean()
bars = axes[2].bar(yr_avg.index, yr_avg.values,
                   color=['#F87171' if y < 2022 else '#3B82F6' for y in yr_avg.index], alpha=0.85)
for bar, val in zip(bars, yr_avg.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[2].set_title('Yıllık Ortalama İş Günü Vaka Hacmi (kırmızı=pandemi)',
                  fontsize=12, fontweight='bold')
axes[2].set_ylabel('Ort. Günlük Vaka'); axes[2].grid(axis='y', alpha=0.3)
daily.drop(columns=['year'], inplace=True, errors='ignore')

# Doluluk–Hacim korelasyonu (gerçek verinin çakıştığı dönem)
overlap = daily[(daily['date'] >= '2022-12-01') & (daily['volume'] > 0)]
sc = axes[3].scatter(overlap['doluluk'], overlap['volume'],
                     c=overlap['date'].map(pd.Timestamp.toordinal),
                     cmap='viridis', alpha=0.4, s=20)
if len(overlap) > 1:
    m_, b_ = np.polyfit(overlap['doluluk'], overlap['volume'], 1)
    x_ = np.linspace(overlap['doluluk'].min(), overlap['doluluk'].max(), 100)
    axes[3].plot(x_, m_*x_+b_, color='#EF4444', lw=2,
                 label=f'Trend: y={m_:.1f}x{b_:+.1f}')
corr = overlap['doluluk'].corr(overlap['volume'])
axes[3].set_title(
    f'Doluluk Oranı ↔ Vaka Hacmi Korelasyonu  (Pearson r = {corr:.3f}, Ara 2022–Kas 2025)',
    fontsize=12, fontweight='bold')
axes[3].set_xlabel('Doluluk Oranı'); axes[3].set_ylabel('Günlük Vaka')
axes[3].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.colorbar(sc, ax=axes[3], label='Zaman (erken→geç)')
axes[3].legend(); axes[3].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'\n📊 Doluluk–Hacim Pearson korelasyonu: r = {corr:.3f}')
print('  Not: Pozitif korelasyon → yüksek doluluk döneminde daha fazla ameliyat beklentisi')

In [ ]:
# CELL 5 — Türk tatil günleri (2019–2026)
# v3: 2020-2022 dini tatilleri eklendi
TR_HOL = set()
for y in range(2019, 2027):
    for m, d in [(1,1),(4,23),(5,1),(5,19),(7,15),(8,30),(10,29)]:
        TR_HOL.add(pd.Timestamp(f'{y}-{m:02d}-{d:02d}'))

RELIGIOUS = {
    # Ramazan Bayramı (Eid al-Fitr)
    pd.Timestamp('2020-05-24'), pd.Timestamp('2020-05-25'), pd.Timestamp('2020-05-26'),  # 2020
    pd.Timestamp('2021-05-13'), pd.Timestamp('2021-05-14'), pd.Timestamp('2021-05-15'),  # 2021
    pd.Timestamp('2022-05-02'), pd.Timestamp('2022-05-03'), pd.Timestamp('2022-05-04'),  # 2022
    pd.Timestamp('2023-04-21'), pd.Timestamp('2023-04-22'), pd.Timestamp('2023-04-23'),  # 2023
    pd.Timestamp('2024-04-10'), pd.Timestamp('2024-04-11'), pd.Timestamp('2024-04-12'),  # 2024
    pd.Timestamp('2025-03-30'), pd.Timestamp('2025-03-31'), pd.Timestamp('2025-04-01'),  # 2025
    pd.Timestamp('2026-03-20'), pd.Timestamp('2026-03-21'), pd.Timestamp('2026-03-22'),  # 2026
    # Kurban Bayramı (Eid al-Adha)
    pd.Timestamp('2020-07-31'), pd.Timestamp('2020-08-01'), pd.Timestamp('2020-08-02'),  # 2020
    pd.Timestamp('2021-07-20'), pd.Timestamp('2021-07-21'), pd.Timestamp('2021-07-22'),  # 2021
    pd.Timestamp('2022-07-09'), pd.Timestamp('2022-07-10'), pd.Timestamp('2022-07-11'),  # 2022
    pd.Timestamp('2023-06-28'), pd.Timestamp('2023-06-29'), pd.Timestamp('2023-06-30'),  # 2023
    pd.Timestamp('2024-06-17'), pd.Timestamp('2024-06-18'), pd.Timestamp('2024-06-19'),  # 2024
    pd.Timestamp('2025-06-06'), pd.Timestamp('2025-06-07'), pd.Timestamp('2025-06-08'),  # 2025
}
ALL_HOL = TR_HOL | RELIGIOUS
print(f'✔ {len(ALL_HOL)} tatil günü kayıt edildi (2019–2026, ulusal + dini)')

In [ ]:
# CELL 6 — Özellik mühendisliği
# v5: HOL_WINDOW parametresi, lag {3,60,90}, EWMA doluluk, COVID_END parametresi

def engineer_features(df):
    df = df.copy()
    d  = pd.to_datetime(df['date'])

    # Zaman bileşenleri
    df['day_of_year']  = d.dt.dayofyear
    df['week_of_year'] = d.dt.isocalendar().week.astype(int)
    df['year']         = d.dt.year

    # Döngüsel kodlama
    for feat, period in [('dayofweek', 7), ('month', 12), ('quarter', 4)]:
        v = getattr(d.dt, feat)
        df[f'sin_{feat}'] = np.sin(2 * np.pi * v / period)
        df[f'cos_{feat}'] = np.cos(2 * np.pi * v / period)

    # Tatil & hafta sonu
    df['is_holiday'] = d.isin(ALL_HOL).astype(int)
    df['is_weekend']  = (d.dt.dayofweek >= 5).astype(int)

    # ── v5: HOL_WINDOW parametresi — ±HOL_WINDOW gün tatil yakınlığı ─────────
    for off in range(-HOL_WINDOW, HOL_WINDOW + 1):
        if off == 0: continue
        shifted = {h + pd.Timedelta(days=off) for h in ALL_HOL}
        df[f'near_hol_{off:+d}'] = d.isin(shifted).astype(int)

    # Aylık doktor sayısı
    mk = d.dt.to_period('M')
    pm = prov_monthly.set_index('month_key')['num_prov']
    df['num_prov'] = mk.map(pm).ffill().bfill()

    # ── v5: COVID_END parametresi — pandemi bayrağı ───────────────────────────
    df['is_covid'] = ((d >= '2020-01-01') & (d < COVID_END)).astype(int)

    # Anomali bayrağı
    iso = IsolationForest(contamination=0.05, random_state=SEED)
    df['is_anomaly'] = (iso.fit_predict(df[['volume']]) == -1).astype(int)

    # ── v5: Genişletilmiş lag özellikleri {3, 7, 14, 28, 60, 90} ─────────────
    for lag in [3, 7, 14, 28, 60, 90]:
        df[f'lag_{lag}'] = df['volume'].shift(lag).fillna(0)

    # Geçen yıl aynı hafta (pandemi-farkındalıklı)
    lag_1yr = df['volume'].shift(364)
    lag_2yr = df['volume'].shift(364 * 2)
    lag_3yr = df['volume'].shift(364 * 3)
    df['same_week_ly'] = lag_1yr.fillna(lag_2yr).fillna(lag_3yr).fillna(0)
    covid_mask     = (df['date'] >= '2022-01-01') & (df['date'] < '2024-01-01')
    covid_prev_low = df['same_week_ly'] < 20
    df.loc[covid_mask & covid_prev_low, 'same_week_ly'] = lag_2yr[covid_mask & covid_prev_low].fillna(0)

    # Rolling ortalama
    for win in [7, 14, 28]:
        df[f'roll_{win}'] = (df['volume'].shift(1)
                                        .rolling(win, min_periods=1)
                                        .mean().fillna(0))

    # ── v5: EWMA doluluk — son günlere üstel ağırlık ─────────────────────────
    df['dol_ewma'] = (df['doluluk'].ewm(span=DOL_EWMA_SPAN, adjust=False).mean())

    return df

daily = engineer_features(daily)

# ── FEATURE_COLS: HOL_WINDOW'a göre dinamik tatil sütunları ──────────────────
hol_cols = [f'near_hol_{off:+d}'
            for off in range(-HOL_WINDOW, HOL_WINDOW + 1) if off != 0]

FEATURE_COLS = (
    ['volume',
     'day_of_year', 'week_of_year',
     'sin_dayofweek', 'cos_dayofweek',
     'sin_month',    'cos_month',
     'sin_quarter',  'cos_quarter',
     'is_holiday', 'is_weekend', 'is_anomaly']
    + hol_cols
    + ['num_prov',
       'is_covid',
       'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_60', 'lag_90',   # v5: +lag_3,60,90
       'same_week_ly',
       'roll_7', 'roll_14', 'roll_28',
       'doluluk', 'dol_roll7', 'dol_delta',
       'dol_ewma']                                                    # v5: +EWMA
)

print(f'✔ Özellik mühendisliği tamamlandı — {len(FEATURE_COLS)} özellik')
print(f'  Anomali tespit  : {daily["is_anomaly"].sum()} gün')
print(f'  COVID dönemi    : {daily["is_covid"].sum()} gün (2020 Oca – {COVID_END})')
print(f'  Tatil günü      : {daily["is_holiday"].sum()}')
print(f'  Tatil penceresi : ±{HOL_WINDOW} gün ({len(hol_cols)} sütun)')
print(f'  Doluluk ort.    : {daily["doluluk"].mean():.3f}  |  EWMA ort.: {daily["dol_ewma"].mean():.3f}')
print(f'  Lag seti        : 3, 7, 14, 28, 60, 90')

In [ ]:
# CELL 7 — Eğitim/Validasyon bölünmesi & ölçekleme
# v5: MinMaxScaler → RobustScaler (pandemi aykırılıklarına dayanıklı)

train_df = daily[daily['date'] <  TRAIN_END].copy()   # 2020–2024
val_df   = daily[(daily['date'] >= TRAIN_END) &
                 (daily['date'] <  VAL_END)].copy()    # 2025

scaler   = RobustScaler()          # v5: median & IQR bazlı — aykırı değerlere dayanıklı
train_sc = scaler.fit_transform(train_df[FEATURE_COLS].values)
val_sc   = scaler.transform(val_df[FEATURE_COLS].values)
all_sc   = scaler.transform(daily[FEATURE_COLS].values)

def inv_vol(sv):
    """Ölçeklenmiş hacim değerlerini orijinal ölçeğe geri çevir."""
    d = np.zeros((len(sv), len(FEATURE_COLS)))
    d[:, 0] = sv
    return scaler.inverse_transform(d)[:, 0]

def make_sequences(arr, lb=LOOKBACK, hz=HORIZON):
    X, y = [], []
    for i in range(len(arr) - lb - hz + 1):
        X.append(arr[i : i + lb])
        y.append(arr[i + lb : i + lb + hz, 0])
    return np.array(X), np.array(y)

X_tr, y_tr = make_sequences(train_sc)
X_va, y_va = make_sequences(np.concatenate([train_sc[-LOOKBACK:], val_sc]))

print(f'Eğitim : {len(train_df):,} gün  ({train_df["date"].min().date()} – {train_df["date"].max().date()})')
print(f'Val    : {len(val_df):,} gün   ({val_df["date"].min().date()} – {val_df["date"].max().date()})')
print(f'LSTM dizisi — Eğitim: {len(X_tr):,}  |  Val: {len(X_va):,}')
print(f'Dizi şekli: {X_tr.shape}  →  Hedef şekli: {y_tr.shape}')
print(f'Scaler   : RobustScaler  (center={scaler.center_[0]:.2f}  scale={scaler.scale_[0]:.2f})')

In [ ]:
# CELL 8 — MODEL 1: Encoder-Decoder seq2seq LSTM
# v5: LR=3e-4, clipnorm=5.0, RECURRENT_DROPOUT param, HUBER_DELTA param, 2. decoder katmanı

def build_lstm(n_features):
    enc_in = Input(shape=(LOOKBACK, n_features), name='encoder_input')

    # Bidirectional LSTM — çift yönlü örüntü öğrenimi
    x      = Bidirectional(
                 LSTM(LSTM_UNITS, return_sequences=True,
                      dropout=DROPOUT,
                      recurrent_dropout=RECURRENT_DROPOUT),   # v5: parametre
                 name='bi_lstm'
             )(enc_in)
    _, h, c = LSTM(LSTM_UNITS, return_state=True,
                   dropout=DROPOUT,
                   recurrent_dropout=RECURRENT_DROPOUT,       # v5: parametre
                   name='encoder_lstm')(x)

    # ── v5: Decoder — 2 katmanlı (daha derin kod çözme) ─────────────────────
    dec  = RepeatVector(HORIZON, name='repeat')(h)
    dec  = LSTM(LSTM_UNITS, return_sequences=True,
                dropout=DROPOUT, recurrent_dropout=RECURRENT_DROPOUT,
                name='decoder_lstm_1')(dec, initial_state=[h, c])
    dec  = LSTM(LSTM_UNITS // 2, return_sequences=True,       # v5 YENİ
                dropout=DROPOUT,
                name='decoder_lstm_2')(dec)                   # v5 YENİ
    out  = TimeDistributed(Dense(1), name='output')(dec)
    out  = tf.keras.layers.Reshape((HORIZON,))(out)

    model = Model(enc_in, out, name='seq2seq_bilstm_v5')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,   # v5: 1e-3 → 3e-4
            clipnorm=CLIP_NORM             # v5: 1.0 → 5.0
        ),
        loss=tf.keras.losses.Huber(delta=HUBER_DELTA)   # v5: delta parametresi
    )
    return model

lstm_model = build_lstm(len(FEATURE_COLS))
lstm_model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=EARLY_PATIENCE,
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=REDUCE_FACTOR,
                      patience=REDUCE_PATIENCE, min_lr=1e-7, verbose=1),
]

print(f'\n⏳ LSTM eğitiliyor …')
print(f'   LR={LEARNING_RATE}  clipnorm={CLIP_NORM}  Huber_δ={HUBER_DELTA}')
print(f'   recurrent_dropout={RECURRENT_DROPOUT}  patience={EARLY_PATIENCE}')
print(f'   (GPU ile ~5-12 dk, CPU ile ~40-60 dk)')
history = lstm_model.fit(
    X_tr, y_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=callbacks,
    verbose=1
)
print(f'✔ LSTM eğitimi tamamlandı ({len(history.history["loss"])} epoch)')

# Kayıp eğrisi
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     color='#3B82F6', lw=1.5, label='Eğitim Kaybı')
ax.plot(history.history['val_loss'], color='#F59E0B', lw=1.5, ls='--', label='Val Kaybı')
ax.set_title(f'LSTM — Eğitim & Validasyon Kaybı (Huber δ={HUBER_DELTA})', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Kayıp')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# CELL 9 — LSTM tahminleri
# Direct forecast: her adımda GERÇEK geçmiş veri — hata birikimi yok

def direct_forecast_lstm(mode='val'):
    if mode == 'val':
        target_dates = val_df['date'].values
        n_target     = len(val_df)
    else:
        target_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')
        n_target     = len(target_dates)

    results = []
    dates   = pd.DatetimeIndex(pd.to_datetime(list(target_dates)))

    for i in range(0, n_target, HORIZON):
        chunk = dates[i : i + HORIZON]

        if mode == 'val':
            # Gerçek eğitim + o ana kadarki gerçek val verisi
            real_past = np.vstack([train_sc, val_sc[:i]])
        else:
            # Tüm bilinen gerçek veri (2020-2025)
            real_past = all_sc

        seed  = real_past[-LOOKBACK:]
        x_in  = seed.reshape(1, LOOKBACK, len(FEATURE_COLS))
        ps    = lstm_model.predict(x_in, verbose=0)[0]
        pv    = np.clip(inv_vol(ps), 0, None)

        for dt, p in zip(chunk, pv):
            results.append({'date': pd.Timestamp(dt), 'pred_lstm': round(float(p))})

    return pd.DataFrame(results)

fcast_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')

print('⏳ LSTM validasyon tahmini …')
lstm_val  = direct_forecast_lstm(mode='val')
print('⏳ LSTM 2026 tahmini …')
lstm_test = direct_forecast_lstm(mode='test')
print(f'✔ LSTM: {len(lstm_val)} validasyon, {len(lstm_test)} tahmin')

In [ ]:
# CELL 10 — MODEL 2: Facebook Prophet
# v5: seasonality_mode=multiplicative, yearly_seasonality=15, tüm parametreler CELL 2'den

print('⏳ Prophet eğitiliyor …')

hol_df = pd.DataFrame([
    {'holiday': 'tr_tatil', 'ds': h,
     'lower_window': -2, 'upper_window': 2}
    for h in ALL_HOL
])

def get_prov(dt):
    mk = pd.Period(pd.Timestamp(dt), 'M')
    pm = prov_monthly.set_index('month_key')['num_prov']
    return float(pm.get(mk, pm.iloc[-1]))

def get_doluluk(dt):
    """Tarih için doluluk oranı döndür (impüte edilmiş dol_full serisinden)."""
    ts = pd.Timestamp(dt).normalize()
    return float(dol_full.get(ts, dol_full.mean()))

prophet_model = Prophet(
    seasonality_mode=PROPHET_SEAS_MODE,        # v5: 'multiplicative' (mevsimsel ölçek etkisi)
    weekly_seasonality=True,
    yearly_seasonality=PROPHET_YEARLY_N,       # v5: 10 → 15 Fourier bileşeni
    daily_seasonality=False,
    holidays=hol_df,
    changepoint_prior_scale=PROPHET_CP_SCALE,  # CELL 2 parametresi
    seasonality_prior_scale=PROPHET_SEAS_PRIOR,
    holidays_prior_scale=PROPHET_HOL_PRIOR,
)
prophet_model.add_regressor('num_prov', standardize=True)
prophet_model.add_regressor('is_covid', standardize=False)   # COVID feature
prophet_model.add_regressor('doluluk',  standardize=True)    # ← YENİ v4: doluluk oranı

train_prophet = train_df[['date','volume','is_covid']].rename(
    columns={'date':'ds','volume':'y'}).copy()
train_prophet['num_prov'] = train_prophet['ds'].apply(get_prov)
train_prophet['doluluk']  = train_prophet['ds'].apply(get_doluluk)
prophet_model.fit(train_prophet)

n_future = len(val_df) + len(fcast_dates)
future   = prophet_model.make_future_dataframe(periods=n_future, freq='D')
future['num_prov'] = future['ds'].apply(get_prov)
future['is_covid'] = ((future['ds'] >= '2020-01-01') &
                      (future['ds'] <  '2022-06-01')).astype(int)
future['doluluk']  = future['ds'].apply(get_doluluk)   # ← YENİ v4
fc_prophet = prophet_model.predict(future)

prop_val  = (fc_prophet[fc_prophet['ds'].isin(val_df['date'])]
             [['ds','yhat']]
             .rename(columns={'ds':'date','yhat':'pred_prophet'}))
prop_val['pred_prophet'] = np.clip(prop_val['pred_prophet'], 0, None).round()

prop_test = (fc_prophet[fc_prophet['ds'].isin(fcast_dates)]
             [['ds','yhat']]
             .rename(columns={'ds':'date','yhat':'pred_prophet'}))
prop_test['pred_prophet'] = np.clip(prop_test['pred_prophet'], 0, None).round()

print('✔ Prophet tamamlandı')

# Bileşen grafiği
fig = prophet_model.plot_components(fc_prophet)
fig.suptitle('Prophet — Trend & Mevsimsellik Bileşenleri (2020–2026)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# CELL 11 — MODEL 3: LightGBM
# v5: sklearn GradientBoostingRegressor → LightGBM (10× hız, daha iyi performans)

print('⏳ LightGBM eğitiliyor …')

def make_seq_flat(arr, lb=LOOKBACK, hz=HORIZON):
    X, y = [], []
    for i in range(len(arr) - lb - hz + 1):
        X.append(arr[i : i + lb].flatten())
        y.append(arr[i + lb : i + lb + hz, 0])
    return np.array(X), np.array(y)

X_tr_f, y_tr_f = make_seq_flat(train_sc)

# LightGBM: her HORIZON adımı için ayrı model (MultiOutputRegressor eşdeğeri)
lgb_models = []
for step in range(HORIZON):
    m = lgb.LGBMRegressor(
        n_estimators     = LGB_N_ESTIMATORS,
        max_depth        = LGB_MAX_DEPTH,
        learning_rate    = LGB_LR,
        subsample        = LGB_SUBSAMPLE,
        subsample_freq   = 1,
        colsample_bytree = LGB_COLSAMPLE,     # v5 YENİ
        min_child_samples= LGB_MIN_LEAF,
        random_state     = SEED,
        n_jobs           = -1,
        verbose          = -1
    )
    m.fit(X_tr_f, y_tr_f[:, step])
    lgb_models.append(m)

print(f'✔ LightGBM tamamlandı ({HORIZON} model × {LGB_N_ESTIMATORS} ağaç)')
print(f'   depth={LGB_MAX_DEPTH}  lr={LGB_LR}  subsample={LGB_SUBSAMPLE}  col={LGB_COLSAMPLE}')

def direct_forecast_lgb(mode='val'):
    if mode == 'val':
        target_dates = val_df['date'].values
        n_target     = len(val_df)
    else:
        target_dates = fcast_dates
        n_target     = len(fcast_dates)

    results = []
    dates   = pd.DatetimeIndex(pd.to_datetime(list(target_dates)))

    for i in range(0, n_target, HORIZON):
        chunk = dates[i : i + HORIZON]
        if mode == 'val':
            real_past = np.vstack([train_sc, val_sc[:i]])
        else:
            real_past = all_sc
        seed = real_past[-LOOKBACK:]
        flat = seed.flatten().reshape(1, -1)
        ps   = np.array([m.predict(flat)[0] for m in lgb_models])
        pv   = np.clip(inv_vol(ps), 0, None)
        for dt, p in zip(chunk, pv):
            results.append({'date': pd.Timestamp(dt), 'pred_lgb': round(float(p))})

    return pd.DataFrame(results)

gbm_val  = direct_forecast_lgb(mode='val')
gbm_test = direct_forecast_lgb(mode='test')
# Sütun adı uyumluluğu için alias ekle (Cell 12 pred_gbm sütununu okur)
gbm_val['pred_gbm']  = gbm_val['pred_lgb']
gbm_test['pred_gbm'] = gbm_test['pred_lgb']
print(f'✔ LightGBM: {len(gbm_val)} validasyon, {len(gbm_test)} tahmin')

In [ ]:
# CELL 12 — Metrikler & karşılaştırma + Weighted Ensemble
# v5: val R²'ye göre ağırlıklı ensemble eklendi

def compute_metrics(y_true, y_pred):
    yt = np.array(y_true); yp = np.array(y_pred)
    m  = yt > 0
    return (r2_score(yt[m], yp[m]),
            np.sqrt(mean_squared_error(yt[m], yp[m])),
            mean_absolute_error(yt[m], yp[m]))

def build_comparison(actual_df, *pred_dfs):
    merged = actual_df[['date','volume']].copy()
    for pdf in pred_dfs:
        merged = merged.merge(pdf, on='date', how='left')
    return merged.fillna(0)

val_comp = build_comparison(val_df, lstm_val, prop_val, gbm_val)

# Gerçek 2026 verisi (dosyada varsa kullan)
actual_2026 = daily[daily['date'] >= pd.Timestamp(FORECAST_START)][['date','volume']]

test_comp = pd.DataFrame({'date': fcast_dates})
for pdf in [lstm_test, prop_test, gbm_test]:
    test_comp = test_comp.merge(pdf, on='date', how='left')
test_comp = test_comp.merge(actual_2026, on='date', how='left')
test_comp = test_comp.fillna(0)
test_comp['is_weekend'] = pd.DatetimeIndex(test_comp['date']).dayofweek >= 5
test_comp['is_holiday'] = pd.DatetimeIndex(test_comp['date']).isin(ALL_HOL)

# Metrik tablosu
print('='*62)
print(f"  Validasyon Performansı — 2025")
print(f"  {'Model':<16} {'R²':>6} {'RMSE':>7} {'MAE':>7}  {'Durum':<12}")
print('  ' + '─'*56)
for col, name in [('pred_lstm','LSTM'),('pred_prophet','Prophet'),('pred_gbm','LightGBM')]:
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    durum = '✅ İyi' if r2 >= 0.7 else ('⚠️ Orta' if r2 >= 0.4 else '❌ Düşük')
    print(f"  {name:<16} {r2:>6.3f} {rm:>7.2f} {ma:>7.2f}  {durum}")
print(f"  {'Paper LSTM':<16} {'0.855':>6} {'2.017':>7} {'1.104':>7}  (referans)")
print('='*62)
# ── v5: Ağırlıklı Ensemble — Val R²'ye göre ──────────────────────────────────
print()
print('='*62)
print('  Ağırlıklı Ensemble (val R²)')

R2_lstm    = compute_metrics(val_comp['volume'], val_comp['pred_lstm'])[0]
R2_prophet = compute_metrics(val_comp['volume'], val_comp['pred_prophet'])[0]
R2_gbm     = compute_metrics(val_comp['volume'], val_comp['pred_gbm'])[0]

# Negatif R²'yi 0'a çek (katkısız model)
w_raw = np.array([max(R2_lstm, 0), max(R2_prophet, 0), max(R2_gbm, 0)])
if w_raw.sum() == 0:
    w_raw = np.ones(3)
weights   = w_raw / w_raw.sum()
w_lstm, w_prophet, w_gbm = weights

print(f'  Ağırlıklar: LSTM={w_lstm:.3f}  Prophet={w_prophet:.3f}  LightGBM={w_gbm:.3f}')

val_comp['pred_ensemble'] = (
    w_lstm    * val_comp['pred_lstm'] +
    w_prophet * val_comp['pred_prophet'] +
    w_gbm     * val_comp['pred_gbm']
).round()
test_comp['pred_ensemble'] = (
    w_lstm    * test_comp['pred_lstm'] +
    w_prophet * test_comp['pred_prophet'] +
    w_gbm     * test_comp['pred_gbm']
).round()

r2e, rme, mae_e = compute_metrics(val_comp['volume'], val_comp['pred_ensemble'])
durum_e = '✅ İyi' if r2e >= 0.7 else ('⚠️ Orta' if r2e >= 0.4 else '❌ Düşük')
print(f'  Ensemble         {r2e:>6.3f} {rme:>7.2f} {mae_e:>7.2f}  {durum_e}')
print('='*62)


# Gerçek 2026 verisi varsa karşılaştır
has_actual = test_comp['volume'].sum() > 0
if has_actual:
    print()
    print('='*62)
    print(f"  Gerçek Karşılaştırma — Jan-Mar 2026 (mevcut veri)")
    print(f"  {'Model':<16} {'R²':>6} {'RMSE':>7} {'MAE':>7}")
    print('  ' + '─'*42)
    for col, name in [('pred_lstm','LSTM'),('pred_prophet','Prophet'),('pred_gbm','LightGBM'),('pred_ensemble','Ensemble')]:
        r2, rm, ma = compute_metrics(test_comp['volume'], test_comp[col])
        print(f"  {name:<16} {r2:>6.3f} {rm:>7.2f} {ma:>7.2f}")
    print('='*62)

# Ocak 2026 tablosu
jan = test_comp[pd.DatetimeIndex(test_comp['date']).month == 1].copy()
jan['day'] = pd.DatetimeIndex(jan['date']).day_name()
print(f'\n📅 Ocak 2026 Tahminleri')
header = f"{'Tarih':<13} {'Gün':<12} {'LSTM':>6} {'Prophet':>8} {'GBM':>6}"
if has_actual: header += f" {'Gerçek':>8}"
header += '  Not'
print(header); print('─'*70)
for _, r in jan.iterrows():
    note = '🏥 Tatil' if r['is_holiday'] else ('🔵 Hft.sonu' if r['is_weekend'] else '')
    row = (f"{str(r['date'].date()):<13} {r['day']:<12} "
           f"{int(r['pred_lstm']):>6} {int(r['pred_prophet']):>8} {int(r['pred_gbm']):>6} {int(r['pred_ensemble']):>6}")
    if has_actual: row += f" {int(r['volume']):>8}"
    print(row + f'  {note}')
print('─'*70)
wdays = jan[~jan['is_weekend'] & ~jan['is_holiday']]
for col, name in [('pred_lstm','LSTM'),('pred_prophet','Prophet'),('pred_gbm','LightGBM'),('pred_ensemble','Ensemble')]:
    print(f'  {name:<12} Ocak toplamı: {jan[col].sum():.0f}  |  İş günü ort: {wdays[col].mean():.1f}')

In [ ]:
# CELL 13 — Ana görselleştirme (5 satır)

COLS = {'LSTM':'#3B82F6', 'Prophet':'#F59E0B', 'GBM':'#10B981', 'Ensemble':'#8B5CF6', 'Actual':'#1E293B'}
has_actual = test_comp['volume'].sum() > 0

fig = plt.figure(figsize=(18, 34), facecolor='#F8FAFC')
gs  = gridspec.GridSpec(6, 3, figure=fig, hspace=0.52, wspace=0.32,
                        top=0.95, bottom=0.03, left=0.06, right=0.97)

# ── Satır 0: Validasyon 2025
ax0 = fig.add_subplot(gs[0, :]); ax0.set_facecolor('white')
ax0.plot(val_df['date'], val_df['volume'],
         color=COLS['Actual'], lw=2, label='Gerçek', zorder=5)
for col, name, ls in [('pred_lstm','LSTM','--'),
                       ('pred_prophet','Prophet',':'),
                       ('pred_gbm','LightGBM','-.'),
                       ('pred_ensemble','Ensemble','-')]:
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    ax0.plot(val_df['date'], val_comp[col], color=COLS.get(name, COLS['GBM']), lw=1.7, ls=ls.strip(),
             label=f'{name:<12} R²={r2:.3f}  RMSE={rm:.1f}  MAE={ma:.1f}')
ax0.set_title('Validasyon — 2025: Tüm Modeller + Ensemble vs Gerçek Günlük Vaka',
              fontsize=12, fontweight='bold')
ax0.set_ylabel('Günlük Vaka')
ax0.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax0.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax0.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax0.legend(fontsize=9, ncol=2, loc='upper left', prop={'family':'monospace'})
ax0.grid(alpha=0.2, lw=0.5)

# ── Satır 1: Ocak-Mart 2026 Tahminleri
ax1 = fig.add_subplot(gs[1, :]); ax1.set_facecolor('white')
for _, r in test_comp[test_comp['is_weekend'] | test_comp['is_holiday']].iterrows():
    ax1.axvspan(r['date'] - pd.Timedelta(hours=12),
                r['date'] + pd.Timedelta(hours=12),
                alpha=0.07, color='gray')
if has_actual:
    ax1.bar(test_comp['date'], test_comp['volume'],
            color='#CBD5E1', width=0.85, label='Gerçek', zorder=2)
for col, name, ls in [('pred_lstm','LSTM','--'),
                       ('pred_prophet','Prophet',':'),
                       ('pred_gbm','LightGBM','-.'),
                       ('pred_ensemble','Ensemble','-')]:
    lbl = name
    if has_actual:
        r2, rm, ma = compute_metrics(test_comp['volume'], test_comp[col])
        lbl = f'{name}  R²={r2:.3f}  MAE={ma:.1f}'
    ax1.plot(test_comp['date'], test_comp[col], color=COLS.get(name, COLS['GBM']),
             lw=2.2, ls=ls.strip(), label=lbl, zorder=5)
ax1.set_title('Tahmin — Ocak–Mart 2026 (gri = hafta sonu/tatil)',
              fontsize=12, fontweight='bold')
ax1.set_ylabel('Günlük Vaka')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax1.legend(fontsize=9, ncol=4, prop={'family':'monospace'})
ax1.grid(alpha=0.2, lw=0.5)

# ── Satır 2: Scatter (validasyon)
for ci, (col, name) in enumerate([('pred_lstm','LSTM'),
                                   ('pred_prophet','Prophet'),
                                   ('pred_gbm','LightGBM')]):
    ax = fig.add_subplot(gs[2, ci]); ax.set_facecolor('white')
    mw = val_comp[val_comp['volume'] > 0]
    ax.scatter(mw['volume'], mw[col], color=COLS[name],
               alpha=0.55, s=35, zorder=3, edgecolors='white', lw=0.5)
    mn = max(0, min(mw['volume'].min(), mw[col].min()) - 3)
    mx = max(mw['volume'].max(), mw[col].max()) + 3
    ax.plot([mn,mx],[mn,mx], '--', color='#94A3B8', lw=1.2, label='Mükemmel fit')
    if len(mw) > 1:
        m_, b_ = np.polyfit(mw['volume'], mw[col], 1)
        ax.plot([mn,mx],[m_*mn+b_, m_*mx+b_], '-', color='#EF4444',
                lw=1.5, alpha=0.7, label=f'y={m_:.2f}x{b_:+.0f}')
    r2, rm, ma = compute_metrics(mw['volume'], mw[col])
    ax.set_xlabel('Gerçek Vaka', fontsize=9); ax.set_ylabel('Tahmin', fontsize=9)
    ax.set_title(f'{name} — Validasyon 2025', fontsize=10, fontweight='bold')
    ax.text(0.06, 0.83, f'R²   = {r2:.3f}\nRMSE = {rm:.1f}\nMAE  = {ma:.1f}',
            transform=ax.transAxes, fontsize=9, family='monospace',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#DBEAFE', alpha=0.85))
    ax.legend(fontsize=8); ax.grid(alpha=0.2, lw=0.5)

# ── Satır 3: Hata dağılımı
for ci, (col, name) in enumerate([('pred_lstm','LSTM'),
                                   ('pred_prophet','Prophet'),
                                   ('pred_gbm','LightGBM')]):
    ax = fig.add_subplot(gs[3, ci]); ax.set_facecolor('white')
    mw  = val_comp[val_comp['volume'] > 0]
    err = mw[col].values - mw['volume'].values
    ax.hist(err, bins=28, color=COLS[name], alpha=0.75, edgecolor='white')
    ax.axvline(0,          color='black',   lw=1.5, ls='--', label='Sıfır hata')
    ax.axvline(err.mean(), color='#EF4444', lw=1.5, label=f'Ort={err.mean():+.1f}')
    ax.set_title(f'{name} — Hata Dağılımı (Validasyon 2025)',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Hata (Tahmin − Gerçek)'); ax.set_ylabel('Sayı')
    ax.legend(fontsize=8); ax.grid(alpha=0.2, lw=0.5)

# ── Satır 4: Performans karşılaştırma
ax4 = fig.add_subplot(gs[4, :]); ax4.set_facecolor('white')
mnames = ['LSTM\n(Enc-Dec)', 'Prophet\n(Facebook)', 'LightGBM', 'Ensemble\n(weighted)']
x, w = np.arange(4), 0.18
R2s = [compute_metrics(val_comp['volume'], val_comp[c])[0]
       for c in ['pred_lstm','pred_prophet','pred_gbm','pred_ensemble']]
RMs = [compute_metrics(val_comp['volume'], val_comp[c])[1]
       for c in ['pred_lstm','pred_prophet','pred_gbm','pred_ensemble']]
MAs = [compute_metrics(val_comp['volume'], val_comp[c])[2]
       for c in ['pred_lstm','pred_prophet','pred_gbm','pred_ensemble']]
b1 = ax4.bar(x-w*1.5, R2s,                  w, color='#3B82F6', alpha=0.85, label='R²', zorder=3)
b2 = ax4.bar(x-w*0.5, [r/100 for r in RMs], w, color='#EF4444', alpha=0.85, label='RMSE ÷ 100', zorder=3)
b3 = ax4.bar(x+w*0.5, [m/100 for m in MAs], w, color='#10B981', alpha=0.85, label='MAE ÷ 100',  zorder=3)
for b, v in zip(b1, R2s): ax4.text(b.get_x()+b.get_width()/2, b.get_height()+0.008, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold', color='#1E3A8A')
for b, v in zip(b2, RMs): ax4.text(b.get_x()+b.get_width()/2, b.get_height()+0.008, f'{v:.1f}',  ha='center', fontsize=10, color='#B91C1C')
for b, v in zip(b3, MAs): ax4.text(b.get_x()+b.get_width()/2, b.get_height()+0.008, f'{v:.1f}',  ha='center', fontsize=10, color='#065F46')
ax4.axhline(0.855, color='#3B82F6', lw=1.5, ls=':', alpha=0.5,
            label='Makale LSTM R² kıyası = 0.855')
ax4.set_xticks(x); ax4.set_xticklabels(mnames, fontsize=12)
ax4.set_title('Model Performans Karşılaştırması — Validasyon 2025',
              fontsize=12, fontweight='bold')
ax4.set_ylim(0, 1.15); ax4.legend(fontsize=9, ncol=4)
ax4.grid(axis='y', alpha=0.2, lw=0.5)
best_idx  = int(np.argmax(R2s))
best_name = ['LSTM','Prophet','LightGBM','Ensemble'][best_idx]
ax4.text(0.5, 0.91,
         f'🏆  En iyi model: {best_name}  (R² = {R2s[best_idx]:.3f})',
         transform=ax4.transAxes, ha='center', fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF3C7', alpha=0.9))

# ── Satır 5: Doluluk Oranı — Zaman serisi (ikiz eksen) + korelasyon scatter ──
ax5L = fig.add_subplot(gs[5, :2]); ax5L.set_facecolor('white')
ax5R = ax5L.twinx()

dol_vis  = dol_full[(dol_full.index >= daily['date'].min()) &
                    (dol_full.index <= daily['date'].max())]

ax5L.fill_between(dol_vis.index, dol_vis.values, alpha=0.15, color='#8B5CF6')
ax5L.plot(dol_vis.index, dol_vis.values,
          color='#8B5CF6', lw=0.8, alpha=0.7, label='Doluluk (tüm)')
ax5L.plot(df_dol['doluluk'].index, df_dol['doluluk'].values,
          color='#4C1D95', lw=1.4, label='Doluluk (gerçek)')
ax5R.plot(daily['date'], daily['volume'],
          color='#3B82F6', lw=0.6, alpha=0.45, label='Vaka Hacmi')
ax5L.set_ylabel('Doluluk Oranı', color='#4C1D95', fontsize=9)
ax5R.set_ylabel('Günlük Vaka',   color='#3B82F6', fontsize=9)
ax5L.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax5L.set_title('Doluluk Oranı & Vaka Hacmi — Birlikte Seyir (v4 yeni)',
               fontsize=10, fontweight='bold')
lines1, labels1 = ax5L.get_legend_handles_labels()
lines2, labels2 = ax5R.get_legend_handles_labels()
ax5L.legend(lines1+lines2, labels1+labels2, fontsize=8, loc='upper left')
ax5L.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax5L.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
plt.setp(ax5L.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax5L.grid(alpha=0.2, lw=0.5)

ax5S = fig.add_subplot(gs[5, 2]); ax5S.set_facecolor('white')
overlap = daily[(daily['date'] >= '2022-12-01') & (daily['volume'] > 0)].copy()
if len(overlap) > 10:
    sc5 = ax5S.scatter(overlap['doluluk'], overlap['volume'],
                       c=overlap['date'].map(pd.Timestamp.toordinal),
                       cmap='plasma', alpha=0.35, s=18, zorder=3)
    m5, b5 = np.polyfit(overlap['doluluk'], overlap['volume'], 1)
    x5 = np.linspace(overlap['doluluk'].min(), overlap['doluluk'].max(), 100)
    ax5S.plot(x5, m5*x5+b5, color='#EF4444', lw=2,
              label=f'Trend: y={m5:.1f}x{b5:+.1f}')
    corr5 = overlap['doluluk'].corr(overlap['volume'])
    plt.colorbar(sc5, ax=ax5S, label='Zaman →')
    ax5S.text(0.05, 0.88, f'Pearson r = {corr5:.3f}',
              transform=ax5S.transAxes, fontsize=10, fontweight='bold',
              bbox=dict(boxstyle='round,pad=0.4', facecolor='#EDE9FE', alpha=0.9))
ax5S.set_xlabel('Doluluk Oranı', fontsize=9)
ax5S.set_ylabel('Günlük Vaka',   fontsize=9)
ax5S.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax5S.set_title('Doluluk ↔ Vaka Korelasyonu', fontsize=10, fontweight='bold')
if len(overlap) > 10: ax5S.legend(fontsize=8)
ax5S.grid(alpha=0.2, lw=0.5)

fig.suptitle(
    'KUH Hastanesi – Ameliyat Vaka Hacmi Tahmini (2020–2026 Veri Seti) — v5\n'
    'LSTM  ·  Prophet  ·  LightGBM  ·  Weighted Ensemble  ·  Doluluk Oranı  |  Bui et al. (2023)',
    fontsize=13, fontweight='bold', y=0.98
)
plt.savefig('kuh_forecast_v5.png', dpi=150, bbox_inches='tight', facecolor='#F8FAFC')
plt.show()
print('✅ Grafik kaydedildi: kuh_forecast_v5.png')

In [ ]:
# CELL 14 — Kaydet & İndir
from google.colab import files
import pickle

lstm_model.save('kuh_lstm_v5_tuned.keras')

with open('kuh_lgb_scaler_v5.pkl', 'wb') as f:
    pickle.dump({'lgb_models': lgb_models, 'scaler': scaler,
                 'feature_cols': FEATURE_COLS,
                 'weights': {'lstm': w_lstm, 'prophet': w_prophet, 'lgb': w_gbm}}, f)

test_comp.to_csv('kuh_tahmin_oca_mar_2026.csv', index=False)
val_comp.to_csv('kuh_validasyon_2025.csv', index=False)

for fname in ['kuh_forecast_v5.png',
              'kuh_tahmin_oca_mar_2026.csv',
              'kuh_validasyon_2025.csv',
              'kuh_lstm_v5_tuned.keras']:
    try:
        files.download(fname)
    except Exception as e:
        print(f'  ⚠️ {fname}: {e}')

print('\n✅ Tüm dosyalar indirildi!')
print('  📊 kuh_forecast_v5.png                — ana grafik (ensemble dahil)')
print('  📋 kuh_tahmin_oca_mar_2026.csv        — Oca-Mar 2026 tahminleri + ensemble')
print('  📋 kuh_validasyon_2025.csv            — 2025 validasyon sonuçları')
print('  🤖 kuh_lstm_v5_tuned.keras            — eğitilmiş LSTM modeli (v5)')
print('  📦 kuh_lgb_scaler_v5.pkl              — LightGBM modelleri + ensemble ağırlıkları')